# 02d — Mejorar la A3: elegir mejor el checkpoint y entrenar con negativos difíciles

El diagnóstico de errores de la A3 de Holes (`reports/cisutac/errores/README.md`, issue #5) encontró que el mAP@0.5 bajo (0,131 en valid) **no se debe a que el modelo no vea los agujeros**: entre sus 100 predicciones más confiables por foto llega al 87 %. El problema es que la confianza no separa los aciertos de dos errores:

1. **alucinaciones** sobre texturas que parecen agujeros: ojalillos, lunares, ranuras, pliegues (Δ mAP +0,120);
2. agujeros encontrados con **cajas mal encuadradas** y confianza baja (Δ mAP +0,148).

Además mostró que sumar más tiles sanos **sorteados al azar** (neg 3) no reduce las alucinaciones. Este notebook prueba, sólo en **valid**, dos cambios simples que apuntan a esa causa:

| ID | Cambio | Hipótesis |
|---|---|---|
| **M1** | Elegir el checkpoint con el **mAP@0.5 por foto** (SAHI sobre las fotos de valid) en vez del `best.pt` de Ultralytics | Ultralytics elige la época mirando 162 tiles de valid, sólo 81 de los 1.270 tiles sanos: no ve dónde aparecen las alucinaciones |
| **M2** | Entrenar con **negativos difíciles**: los tiles sanos de train donde la A3 más se dispara, en vez de sorteados | El modelo aprende a rechazar justo las texturas que confunde. Misma *cantidad* de negativos que la A3: cambia sólo *cuáles* |

**Reglas:** misma semilla (0), mismo YOLO11n, mismo presupuesto de imágenes vistas que la A3 del 02 y la misma evaluación por foto en coordenadas de la original. **Test sigue cerrado**: la variante que gane acá se evalúa después en test y con tres semillas (como en el 02c).

> **Negativo difícil** (*hard negative mining*): en vez de mostrarle al modelo ejemplos sanos cualquiera, se le muestran los sanos que hoy confunde con un defecto. Es una idea clásica de detección (Viola-Jones, Felzenszwalb et al.) y del entrenamiento de SSD (OHEM).

## 0. Parámetros

In [ ]:
DATASET = "holes"        # "holes" o "spots"
MODO_PRUEBA = True       # True: 1 época y pocas imágenes, sólo para verificar el pipeline

MODELO = "yolo11n.pt"
SEMILLA = 0
EPOCAS_BASE, PACIENCIA_BASE = 100, 30   # presupuesto de la foto completa; los tiles se igualan en imágenes vistas
TILE, SOLAPAMIENTO, MIN_VISIBLE = 640, 0.2, 0.5
NEG_POR_POS = 1.0                        # misma cantidad de negativos que la A3 de Holes
BATCH = 16
CONF_EVAL = 0.001
PERIODO_CHECKPOINT = 5                   # guarda un checkpoint cada 5 épocas (M1)

# M1: además de la A3 con presupuesto igualado, una A3 extendida (como la 02b) donde la elección de época pesa más
EPOCAS_EXTENDIDA, PACIENCIA_EXTENDIDA = 100, 30
# M2: fracción de los negativos que son difíciles (el resto se sortea como en la A3)
FRACCIONES_DIFICILES = [1.0, 0.5]
CONF_MINERIA = 0.01                      # confianza mínima para puntuar un tile sano

if MODO_PRUEBA:
    EPOCAS_BASE, PACIENCIA_BASE, EPOCAS_EXTENDIDA, PACIENCIA_EXTENDIDA, PERIODO_CHECKPOINT = 1, 1, 2, 1, 1
    LIMITE_EVAL, MAX_IMAGENES_TILES = 3, 6
else:
    LIMITE_EVAL, MAX_IMAGENES_TILES = None, None
SUFIJO = "_prueba" if MODO_PRUEBA else ""

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
import cisutac_baseline as cb
import analyze_detection_errors as ade

pd.set_option("display.max_columns", 30)
print("Datos:", cb.DATA)
print("Resultados:", cb.RESULTS)
print("Dispositivo:", cb.pick_device())

if not (cb.derived_dir(DATASET) / "data.yaml").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_cisutac_yolo.py")], check=True)
n_completas = MAX_IMAGENES_TILES or len(cb.load_ground_truth(DATASET, "train")["images"])
SALIDA = cb.RESULTS / "mejora_a3" / f"{DATASET}{SUFIJO}"
SALIDA.mkdir(parents=True, exist_ok=True)


def nombre(variante):
    return f"{DATASET}_{variante}{SUFIJO}"


def presupuesto(stats):
    # Igual que el 02: épocas y paciencia equivalentes en imágenes vistas a EPOCAS_BASE sobre fotos completas.
    epocas = cb.equivalent_epochs(EPOCAS_BASE, n_completas, stats["train"]["tiles_totales"],
                                  minimum=1 if MODO_PRUEBA else 10)
    return epocas, max(1, round(PACIENCIA_BASE * epocas / EPOCAS_BASE))


def entrenar(variante, data_yaml, epocas, paciencia):
    return cb.train(nombre(variante), data_yaml, imgsz=TILE, epochs=epocas, patience=paciencia, batch=BATCH,
                    seed=SEMILLA, model=MODELO, save_period=PERIODO_CHECKPOINT)


def evaluar(run_dir, checkpoint="best"):
    # Evaluación por foto en valid con tiles (SAHI), la misma de todo el bloque A.
    pesos = Path(run_dir) / "weights" / f"{checkpoint}.pt"
    sufijo = "" if checkpoint == "best" else f"__{checkpoint}"
    return cb.evaluate(f"{Path(run_dir).name}{sufijo}__valid_tiles{TILE}_o{round(100 * SOLAPAMIENTO)}", DATASET,
                       pesos, split="valid", mode="tiles", tile=TILE, overlap=SOLAPAMIENTO, conf=CONF_EVAL,
                       limit=LIMITE_EVAL)


def evaluar_checkpoints(run_dir):
    # Evalúa cada epochN.pt guardado por save_period y el best.pt de Ultralytics.
    filas = []
    for p in sorted((Path(run_dir) / "weights").glob("epoch*.pt"), key=lambda p: int(p.stem[5:])):
        m = evaluar(run_dir, p.stem)
        filas.append({"checkpoint": p.stem, "epoca": int(p.stem[5:]), **{k: m[k] for k in METRICAS}})
    m = evaluar(run_dir, "best")
    filas.append({"checkpoint": "best (Ultralytics)", "epoca": None, **{k: m[k] for k in METRICAS}})
    return pd.DataFrame(filas)


METRICAS = ["mAP@0.5", "mAP@0.5:0.95", "AP_small", "precision", "recall", "f1"]

## 1. Datos de tiles y A3 de referencia

La referencia es la A3 del 02 (tiles de 640, 20 % de solapamiento, un tile sano sorteado por cada tile con defecto), reentrenada con la misma semilla pero guardando un checkpoint cada `PERIODO_CHECKPOINT` épocas. Su `best.pt` debería dar un mAP@0.5 por foto parecido al del 02 (0,131 en Holes); si difiere un poco es la variación propia de reentrenar en GPU.

In [ ]:
yaml_azar, stats_azar = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                              seed=SEMILLA, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES)
EPOCAS_TILES, PACIENCIA_TILES = presupuesto(stats_azar)
print(f"Épocas para tiles: {EPOCAS_TILES} (paciencia {PACIENCIA_TILES})")
display(pd.DataFrame(stats_azar).T)

run_base = entrenar(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}_ckpt", yaml_azar, EPOCAS_TILES, PACIENCIA_TILES)
eval_base = evaluar(run_base)
display(cb.metrics_table([eval_base]))

## 2. M1 — Elegir el checkpoint con el mAP@0.5 por foto

Se evalúan, con SAHI sobre las fotos completas de valid, todos los checkpoints guardados de dos corridas:

- la **A3 con presupuesto igualado** (la de la sección 1);
- una **A3 extendida** a `EPOCAS_EXTENDIDA` épocas, como la 02b, donde Ultralytics eligió la última época y el mAP por foto empeoró (0,107).

La curva muestra, época por época, el mAP@0.5 por foto junto con el mAP50 que mira Ultralytics sobre sus 162 tiles. Si las dos curvas no suben y bajan juntas, el criterio de Ultralytics no está midiendo lo que nos importa.

> **Cuidado con el optimismo.** Elegir el mejor de varios checkpoints mirando valid hace que el número de valid de ese checkpoint quede optimista (con 43 fotos). Sirve para decidir el **criterio**; el número que vale es el de test, con varias semillas.

In [ ]:
run_ext = entrenar(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}_ext{EPOCAS_EXTENDIDA}_ckpt", yaml_azar,
                   EPOCAS_EXTENDIDA, PACIENCIA_EXTENDIDA)
checkpoints = {"A3 presupuesto igualado": evaluar_checkpoints(run_base),
               f"A3 extendida ({EPOCAS_EXTENDIDA} épocas)": evaluar_checkpoints(run_ext)}
for etiqueta, tabla in checkpoints.items():
    print(etiqueta)
    display(tabla.round(3))
    tabla.to_csv(SALIDA / f"m1_checkpoints_{etiqueta.split()[1]}.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, len(checkpoints), figsize=(7 * len(checkpoints), 4), squeeze=False)
for ax, (etiqueta, tabla), run in zip(axes[0], checkpoints.items(), (run_base, run_ext)):
    por_epoca = tabla.dropna(subset=["epoca"])
    ax.plot(por_epoca.epoca, por_epoca["mAP@0.5"], "o-", label="mAP@0.5 por foto (SAHI, valid)")
    curvas = cb.training_curves(run)
    if "metrics/mAP50(B)" in curvas:
        ax.plot(curvas["epoch"], curvas["metrics/mAP50(B)"], alpha=0.6, label="mAP50 de Ultralytics (tiles de valid)")
    mejor_best = tabla[tabla.checkpoint.str.startswith("best")]["mAP@0.5"].iloc[0]
    ax.axhline(mejor_best, color="gray", ls="--", label="best.pt de Ultralytics, por foto")
    ax.set_title(etiqueta)
    ax.set_xlabel("época")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(SALIDA / "m1_curvas.png", dpi=150)

In [ ]:
def mejor_checkpoint(tabla):
    por_epoca = tabla.dropna(subset=["epoca"])
    return por_epoca.loc[por_epoca["mAP@0.5"].idxmax()] if len(por_epoca) else None

resumen_m1 = []
for etiqueta, tabla in checkpoints.items():
    best = tabla[tabla.checkpoint.str.startswith("best")].iloc[0]
    elegido = mejor_checkpoint(tabla)
    resumen_m1.append({"corrida": etiqueta, "criterio": "best.pt (Ultralytics)", **best[METRICAS].to_dict()})
    if elegido is not None:
        resumen_m1.append({"corrida": etiqueta, "criterio": f"mAP por foto ({elegido.checkpoint})",
                           **elegido[METRICAS].to_dict()})
resumen_m1 = pd.DataFrame(resumen_m1).set_index(["corrida", "criterio"])
resumen_m1.to_csv(SALIDA / "m1_resumen.csv")
resumen_m1.round(3)

## 3. M2 — Negativos difíciles

1. **Minería.** El `best.pt` de la A3 de referencia se corre sobre **todos** los tiles sanos de train (unos 8.900 en Holes), a la misma escala con la que se entrenó. Cada tile recibe como puntaje la confianza máxima que le da el modelo. Los de mayor puntaje son los que hoy confunde con un defecto.
2. **Nuevo dataset.** Mismos tiles con defecto y **misma cantidad** de tiles sanos que la A3; sólo cambia cuáles. Con fracción 1 todos los sanos son los más difíciles; con 0,5, la mitad son difíciles y la otra mitad se sortea como en la A3. Valid no cambia.
3. **Entrenamiento** con la receta y el presupuesto de la A3, guardando checkpoints, y la misma evaluación por foto.

**Por qué también 0,5.** Si todos los negativos son difíciles, el modelo deja de ver tela sana común y puede volverse inestable. Además, algunos "negativos difíciles" podrían ser **agujeros sin anotar**: enseñarle que no son defectos lo empeoraría. Mezclar mitad y mitad es la opción más conservadora. Abajo se muestran los tiles más difíciles para revisar a ojo si son texturas o defectos sin etiqueta (no se corrige ninguna etiqueta: sólo se informa).

In [ ]:
puntajes = cb.score_negative_tiles(DATASET, run_base / "weights" / "best.pt", tile=TILE, overlap=SOLAPAMIENTO,
                                   min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES, conf=CONF_MINERIA)
serie = pd.Series(puntajes).sort_values(ascending=False)
n_negativos = stats_azar["train"]["tiles_sin_defecto_usados"]
print(f"Tiles sanos puntuados: {len(serie)} · con alguna predicción ≥ {CONF_MINERIA}: {(serie > 0).sum()}")
print(f"Negativos por corrida: {n_negativos} · puntaje del más fácil de los {n_negativos} más difíciles: "
      f"{serie.iloc[min(n_negativos, len(serie)) - 1]:.3f}")
pd.DataFrame({"archivo": [k[0] for k in serie.index], "x0": [k[1] for k in serie.index],
              "y0": [k[2] for k in serie.index], "puntaje": serie.values}).to_csv(
    SALIDA / "m2_puntajes_tiles_sanos.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(serie.values, bins=50)
ax.axvline(serie.iloc[min(n_negativos, len(serie)) - 1], color="C1", ls="--", label="corte de los difíciles")
ax.set_yscale("log")
ax.set_xlabel("confianza máxima de la A3 en el tile sano")
ax.legend()
plt.tight_layout()

In [ ]:
from PIL import Image

def mostrar_tiles(claves, titulo, archivo_salida, n=12):
    fig, axes = plt.subplots(2, n // 2, figsize=(2.2 * n // 2, 4.8))
    for ax, (archivo, x0, y0) in zip(axes.ravel(), claves[:n]):
        with Image.open(cb.original_dir(DATASET) / archivo) as im:
            ax.imshow(im.convert("RGB").crop((x0, y0, x0 + TILE, y0 + TILE)))
        ax.set_title(f"{puntajes[(archivo, x0, y0)]:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.savefig(SALIDA / archivo_salida, dpi=120)

mostrar_tiles(list(serie.index), "Tiles sanos de train donde la A3 más se dispara", "m2_tiles_dificiles.png")

In [ ]:
runs_m2, evals_m2, checkpoints_m2 = {}, [], {}
for fraccion in FRACCIONES_DIFICILES:
    yaml_d, stats_d = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                            seed=SEMILLA, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES,
                                            hard_scores=puntajes, hard_fraction=fraccion)
    assert stats_d["train"]["tiles_totales"] == stats_azar["train"]["tiles_totales"], "cambió el tamaño del dataset"
    run = entrenar(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}_dificiles{fraccion:g}_ckpt", yaml_d,
                   EPOCAS_TILES, PACIENCIA_TILES)
    runs_m2[f"difíciles {fraccion:.0%}"] = run
    evals_m2.append(evaluar(run))
    checkpoints_m2[f"difíciles {fraccion:.0%}"] = evaluar_checkpoints(run)

display(cb.metrics_table([eval_base, *evals_m2]))
display(cb.size_table([eval_base, *evals_m2]))

**¿Bajaron las alucinaciones?** El mismo análisis de errores del issue #5 (TIDE), con el umbral de confianza de valid de cada corrida. Si M2 funciona, el Δ mAP de alucinaciones tiene que bajar respecto de la A3 sin que suban mucho los no detectados.

In [ ]:
corridas = [ade.load_run(cb.RESULTS / "evaluaciones" / m["corrida"]) for m in [eval_base, *evals_m2]]
analisis = [a for a in (ade.analyze(r) for r in corridas if r) if a]
filas_pred, filas_def, filas_delta = ade.table_rows(analisis)
display(pd.DataFrame(filas_pred).set_index("corrida")[["umbral", "predicciones", "pct_aciertos", "pct_localizacion",
                                                       "pct_duplicados", "pct_alucinaciones"]].round(3))
tabla_delta = pd.DataFrame(filas_delta).set_index("corrida").drop(columns="dataset").round(3)
tabla_delta.to_csv(SALIDA / "m2_delta_map.csv")
display(tabla_delta)
ade.plot_summary(analisis, SALIDA / "m2_errores.png")

## 4. Resumen (sólo valid)

Una fila por corrida, con los dos criterios de elección del checkpoint. La columna `mAP@0.5` de `best.pt` es la comparación limpia entre negativos al azar y difíciles (mismo criterio que el 02). La de "mejor checkpoint por foto" es optimista, porque elige mirando valid.

In [ ]:
todas = {"A3 (negativos al azar)": checkpoints["A3 presupuesto igualado"], **checkpoints_m2}
filas = []
for etiqueta, tabla in todas.items():
    best = tabla[tabla.checkpoint.str.startswith("best")].iloc[0]
    elegido = mejor_checkpoint(tabla)
    filas.append({"corrida": etiqueta,
                  "mAP@0.5 best.pt": best["mAP@0.5"], "AP_small best.pt": best["AP_small"],
                  "recall best.pt": best["recall"], "precision best.pt": best["precision"],
                  "mAP@0.5 mejor checkpoint (optimista)": None if elegido is None else elegido["mAP@0.5"],
                  "checkpoint elegido": None if elegido is None else elegido.checkpoint})
resumen = pd.DataFrame(filas).set_index("corrida")
resumen.to_csv(SALIDA / "resumen.csv")
resumen.round(3)

## 5. Conclusiones

> Completar **después** de correr el notebook sin `MODO_PRUEBA`, con los números de las tablas.

- **M1:** ¿el mAP por foto y el mAP50 de Ultralytics suben y bajan juntos? ¿Elegir por mAP por foto cambia la época elegida y cuánto gana (sabiendo que es optimista)?
- **M2:** ¿los negativos difíciles bajan el Δ mAP de alucinaciones y suben el mAP@0.5 de `best.pt` respecto de la A3? ¿Fracción 1 o 0,5?
- **Tiles difíciles:** ¿son texturas (ojalillos, lunares, costuras) o parecen agujeros sin anotar?
- **Siguiente paso:** congelar la variante ganadora y evaluarla en test con tres semillas, como en el 02c. Para saber si la diferencia es mayor que el azar del entrenamiento, compararla con la dispersión entre semillas de la A3 que mide el 02c.